# 12 — Pressure transient analysis (PTA): automatic build-up interpretation

Code: `src/pta.py`. For every shut-in (choke = 0 %):

1. **Build-up data**: per-minute downhole pressure after shut-in; `Pwf` = last flowing pressure before it.
2. **Rate** `q` = **VFM-predicted daily oil rate** on the day before the shut-in (as the challenge asks).
3. **Producing time** `tp = 24·Np/q` (Np = cumulative oil: given cum to 1 Jan 2025 + VFM oil since).
4. **Bourdet derivative** `dP' = d(dP)/d ln(dt_e)`, `dt_e = dt·tp/(tp+dt)`: shows the flow regimes:
   - early **hump** = wellbore storage + skin (fluid still entering the wellbore after the surface valve closes)
   - **flat** = **radial flow** → the only part where the Horner straight line is valid
   - late **decline** = boundary / pressure support effects
5. **Radial-flow window** chosen automatically = flattest part of the derivative after the hump.
6. **Horner line** `Pws = P* − m·log10((tp+dt)/dt)` fitted on that window → `m`, `P*`, `P1hr`.
7. **Properties** (field units):

| | Formula |
|---|---|
| permeability | `k = 162.6·q·B·μ / (m·h)` [md] |
| skin | `s = 1.151·[(P1hr − Pwf)/m − log10(k/(φ·μ·ct·rw²)) + 3.23]` |
| skin pressure drop | `ΔP_skin = 0.869·m·s` [psi] |
| flow efficiency | `FE = (P* − Pwf − ΔP_skin)/(P* − Pwf)` (1 = undamaged) |
| radius of investigation | `r_i = √(k·t/(948·φ·μ·ct))` [ft] |

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import pta

OUT = load.ROOT / 'outputs'
hf = load.load_high_freq_clean()
pvt = load.load_pvt().set_index('well_name')
completion = load.load_completion().set_index('well_name')
vfm_rates = pd.read_excel(OUT / 'VFM_predictions.xlsx', sheet_name='predictions')
vfm_rates['date'] = pd.to_datetime(vfm_rates.date)

events = pta.find_shut_ins(hf)
events

## Analyse every shut-in

In [ ]:
results = []
for _, event in events.iterrows():
    rates = vfm_rates[vfm_rates.well_name == event.well_name]
    day_before = event.t_shut.normalize() - pd.Timedelta(days=1)
    q = rates[rates.date == day_before].oil_rate_stbd.iloc[0]
    cum_oil = completion.loc[event.well_name, 'cum_oil_at_2025_01_01_stb'] + \
              rates[rates.date < event.t_shut.normalize()].oil_rate_stbd.sum()
    results.append(pta.analyze_event(hf, event, q, cum_oil, pvt.loc[event.well_name]))

table = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
table.round(2)

## Planned build-up test or unplanned shut-in?
Textbook: a build-up TEST is designed (planned duration, controlled restart). Signatures: duration ≥ 24 h
and a staged choke ramp-up on reopening (`pta.classify_shut_ins`). Only planned tests are reported as results.

In [ ]:
pta.classify_shut_ins(events, load.load_choke_events())[['well_name', 't_shut', 'duration_hr', 'restart_steps',
                                                        'event_type', 'classification_reason']]

## Plots for every shut-in: (a) history, (b) Horner, (c) log-log derivative

In [ ]:
(OUT / 'plots').mkdir(exist_ok=True)
for r in results:
    fig = pta.plot_event(hf, r, vfm_rates)
    fig.savefig(OUT / 'plots' / f"pta_{r['well_name']}_{r['t_shut']:%Y%m%d}.png", dpi=110)
    plt.show()